In [1]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, classification_report
from imblearn.over_sampling import SMOTE

print("All imports successful!")

All imports successful!


In [2]:
import numpy, pandas
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)

numpy: 2.2.6
pandas: 2.3.3


In [3]:
conn = sqlite3.connect("../data/attrition.db")
df = pd.read_sql("SELECT * FROM employees", conn)
conn.close()

print("Shape:", df.shape)
print("Attrition_Binary distribution:\n", df["Attrition_Binary"].value_counts())

Shape: (1470, 34)
Attrition_Binary distribution:
 Attrition_Binary
0    1233
1     237
Name: count, dtype: int64


In [4]:
# Columns that are text — need to be converted to numbers for the model
label_cols = [
    "BusinessTravel", "Department", "EducationField",
    "Gender", "JobRole", "MaritalStatus", "OverTime"
]

le = LabelEncoder()
df_model = df.copy()

for col in label_cols:
    df_model[col] = le.fit_transform(df_model[col].astype(str))

print("Encoding done. Sample:")
print(df_model[["Department", "JobRole", "OverTime"]].head(3))

Encoding done. Sample:
   Department  JobRole  OverTime
0           2        7         1
1           1        6         0
2           1        2         1


In [5]:
# Feature 1: How does this person's salary compare to others in the same role?
role_avg_income = df_model.groupby("JobRole")["MonthlyIncome"].transform("mean")
df_model["SalaryToRoleRatio"] = (df_model["MonthlyIncome"] / role_avg_income).round(3)

# Feature 2: How stable is their tenure relative to total career?
# (Low score = switched jobs a lot = possible flight risk)
df_model["TenureStabilityScore"] = (
    df_model["YearsAtCompany"] / (df_model["TotalWorkingYears"] + 1)
).round(3)

# Feature 3: Combined satisfaction score across 3 dimensions
df_model["SatisfactionComposite"] = (
    df_model["JobSatisfaction"] +
    df_model["EnvironmentSatisfaction"] +
    df_model["WorkLifeBalance"]
) / 3

print("New features created:")
print(df_model[["SalaryToRoleRatio", "TenureStabilityScore", "SatisfactionComposite"]].describe().round(3))

New features created:
       SalaryToRoleRatio  TenureStabilityScore  SatisfactionComposite
count           1470.000              1470.000               1470.000
mean               1.000                 0.582                  2.737
std                0.345                 0.284                  0.568
min                0.311                 0.000                  1.000
25%                0.743                 0.368                  2.333
50%                0.919                 0.636                  2.667
75%                1.179                 0.833                  3.333
max                3.001                 0.976                  4.000


In [6]:
# Drop columns we don't want as features
drop_cols = ["Attrition", "Attrition_Binary", "EmployeeNumber", "OverTime_Binary"]
drop_cols = [c for c in drop_cols if c in df_model.columns]

X = df_model.drop(columns=drop_cols)
y = df_model["Attrition_Binary"]

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())
print("\nFeature names:")
print(list(X.columns))

Feature matrix shape: (1470, 33)
Target distribution:
 Attrition_Binary
0    1233
1     237
Name: count, dtype: int64

Feature names:
['Age', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'SalaryToRoleRatio', 'TenureStabilityScore', 'SatisfactionComposite']


In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Approach 1: tell the model to weight minority class more heavily
rf_balanced = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
)
auc_balanced = cross_val_score(rf_balanced, X, y, cv=skf, scoring="roc_auc").mean()
print(f"class_weight='balanced'  →  CV AUC: {auc_balanced:.4f}")

# Approach 2: synthetically create more "Yes" samples (SMOTE)
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X, y)
print(f"\nSMOTE resampled: {y_sm.value_counts().to_dict()}")

rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
auc_smote = cross_val_score(rf_smote, X_sm, y_sm, cv=skf, scoring="roc_auc").mean()
print(f"SMOTE                    →  CV AUC: {auc_smote:.4f}")

best_approach = "class_weight" if auc_balanced >= auc_smote else "SMOTE"
print(f"\nWinner: {best_approach}")

class_weight='balanced'  →  CV AUC: 0.8020

SMOTE resampled: {1: 1233, 0: 1233}
SMOTE                    →  CV AUC: 0.9726

Winner: SMOTE


In [8]:
# Logistic Regression baseline
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
auc_lr = cross_val_score(lr, X, y, cv=skf, scoring="roc_auc").mean()
print(f"Logistic Regression baseline  →  CV AUC: {auc_lr:.4f}")
print(f"Random Forest (balanced)      →  CV AUC: {auc_balanced:.4f}")
print(f"RF improvement over LR:  +{(auc_balanced - auc_lr):.4f}")

Logistic Regression baseline  →  CV AUC: 0.7535
Random Forest (balanced)      →  CV AUC: 0.8020
RF improvement over LR:  +0.0485


In [9]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [8, 12, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", "log2"]
}

rf_base = RandomForestClassifier(
    class_weight="balanced", random_state=42, n_jobs=-1
)

grid_search = GridSearchCV(
    rf_base, param_grid,
    cv=skf, scoring="roc_auc",
    n_jobs=-1, verbose=1, refit=True
)

grid_search.fit(X, y)

print("\nBest parameters:", grid_search.best_params_)
print(f"Best CV AUC:     {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_

Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best parameters: {'max_depth': 12, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'n_estimators': 200}
Best CV AUC:     0.8144


In [10]:
y_pred_proba = best_rf.predict_proba(X)[:, 1]
y_pred = best_rf.predict(X)

print(f"Final model AUC: {roc_auc_score(y, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y, y_pred, target_names=["Stayed", "Left"]))

Final model AUC: 0.9949

Classification Report:
              precision    recall  f1-score   support

      Stayed       0.99      0.98      0.99      1233
        Left       0.91      0.95      0.93       237

    accuracy                           0.98      1470
   macro avg       0.95      0.97      0.96      1470
weighted avg       0.98      0.98      0.98      1470



In [11]:
joblib.dump(best_rf, "../output/best_rf_model.pkl")
print("Model saved to outputs/best_rf_model.pkl")

# Also save feature names — you'll need these in Week 3
feature_names = list(X.columns)
pd.Series(feature_names).to_csv("../output/feature_names.csv", index=False)
print("Feature names saved to outputs/feature_names.csv")

Model saved to outputs/best_rf_model.pkl
Feature names saved to outputs/feature_names.csv


In [16]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X)

# Fix: newer versions of SHAP return a 3D array (n_samples, n_features, n_classes)
# We need to handle both old and new formats
if isinstance(shap_values, list):
    # Old format: list of arrays, index [1] = "Left" class
    shap_left = shap_values[1]
elif len(shap_values.shape) == 3:
    # New format: 3D array, last dimension is class, index 1 = "Left"
    shap_left = shap_values[:, :, 1]
else:
    shap_left = shap_values

print("shap_values type:", type(shap_values))
print("shap_left shape:", shap_left.shape)
print("X shape:", X.shape)
print("Shapes match:", shap_left.shape == X.shape)

# --- Plot 1: Global feature importance bar chart ---
plt.figure()
shap.summary_plot(shap_left, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("../output/shap_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved shap_importance.png")

# --- Plot 2: Beeswarm ---
plt.figure()
shap.summary_plot(shap_left, X, show=False)
plt.tight_layout()
plt.savefig("../output/shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved shap_beeswarm.png")

# --- Plot 3: Waterfall for highest-risk employee ---
high_risk_idx = np.argmax(y_pred_proba)
print(f"\nHighest risk employee index: {high_risk_idx}")
print(f"Predicted attrition probability: {y_pred_proba[high_risk_idx]:.1%}")
print(f"Actual outcome: {'LEFT' if y.iloc[high_risk_idx] == 1 else 'STAYED'}")

# Get base value — handle both old and new formats
base_val = explainer.expected_value
if isinstance(base_val, (list, np.ndarray)):
    base_val = base_val[1]

shap_exp = shap.Explanation(
    values=shap_left[high_risk_idx],
    base_values=base_val,
    data=X.iloc[high_risk_idx].values,
    feature_names=X.columns.tolist()
)
plt.figure()
shap.waterfall_plot(shap_exp, show=False)
plt.tight_layout()
plt.savefig("../output/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved shap_waterfall.png")

print("\nWeek 2 complete!")

shap_values type: <class 'numpy.ndarray'>
shap_left shape: (1470, 33)
X shape: (1470, 33)
Shapes match: True
Saved shap_importance.png
Saved shap_beeswarm.png

Highest risk employee index: 688
Predicted attrition probability: 95.0%
Actual outcome: LEFT
Saved shap_waterfall.png

Week 2 complete!
